# Apps Migration: Export Apps to Bundle Structure

This notebook exports apps' source code, generates DAB bundle configs, and scans for catalog references.

**Output per app:**
- Bundle-ready directory with source code and configs
- `catalog_mapping.csv` listing detected catalog references for transformation

## Configuration

In [ ]:
try:
    catalog = dbutils.widgets.get("catalog")
except:
    catalog = "YOUR_SOURCE_CATALOG"

try:
    schema = dbutils.widgets.get("schema")
except:
    schema = "apps_demo"

try:
    volume = dbutils.widgets.get("volume")
except:
    volume = "apps_migration"

try:
    target_host = dbutils.widgets.get("target_host")
except:
    target_host = "https://target-workspace.cloud.databricks.com"

volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
approved_path = f"{volume_path}/apps_inventory_approved/inventory_approved.csv"

print(f"Volume path: {volume_path}")
print(f"Approved inventory: {approved_path}")
print(f"Target host: {target_host}")

## Initialize client

In [ ]:
from databricks.sdk import WorkspaceClient
import json
import base64
from datetime import datetime
from io import StringIO, BytesIO
import csv
import requests
import re

w = WorkspaceClient()
source_host = w.config.host.rstrip('/')
headers = w.config.authenticate()
print(f"Source workspace: {source_host}")

## Read approved inventory

In [ ]:
import pandas as pd

response = w.files.download(approved_path)
content = response.contents.read().decode('utf-8')

approved_df = pd.read_csv(StringIO(content))
print(f"Loaded {len(approved_df)} approved apps")

app_names = approved_df['name'].tolist()
print(f"Apps to export: {app_names}")

## Helper functions

In [ ]:
def get_app_via_rest(app_name):
    resp = requests.get(f"{source_host}/api/2.0/apps/{app_name}", headers=headers)
    if resp.status_code == 200:
        return resp.json()
    else:
        raise Exception(f"Failed to get app {app_name}: {resp.text}")

def export_workspace_file(workspace_path):
    """Export a single file from workspace, return content as bytes."""
    resp = requests.get(
        f"{source_host}/api/2.0/workspace/export",
        headers=headers,
        params={"path": workspace_path, "format": "AUTO"}
    )
    if resp.status_code == 200:
        return base64.b64decode(resp.json().get('content', ''))
    return None

def list_workspace_dir(workspace_path):
    """List files in a workspace directory."""
    resp = requests.get(
        f"{source_host}/api/2.0/workspace/list",
        headers=headers,
        params={"path": workspace_path}
    )
    if resp.status_code == 200:
        return resp.json().get('objects', [])
    return []

def safe_name(name):
    """Convert app name to safe bundle resource name."""
    return re.sub(r'[^a-zA-Z0-9_]', '_', name)

def scan_for_catalogs(content, filename):
    """Scan file content for catalog references. Returns list of (catalog, context)."""
    catalogs_found = []
    
    if not content:
        return catalogs_found
    
    try:
        text = content.decode('utf-8') if isinstance(content, bytes) else content
    except:
        return catalogs_found
    
    # Pattern: catalog.schema.table or catalog.schema
    # Matches: spark.table("catalog.schema.table"), FROM catalog.schema.table, etc.
    patterns = [
        # Three-part: catalog.schema.table
        r'["\']([a-zA-Z_][a-zA-Z0-9_]*)\.([a-zA-Z_][a-zA-Z0-9_]*)\.([a-zA-Z_][a-zA-Z0-9_]*)["\']',
        # SQL FROM/JOIN/INTO: FROM catalog.schema.table
        r'(?:FROM|JOIN|INTO|TABLE)\s+([a-zA-Z_][a-zA-Z0-9_]*)\.([a-zA-Z_][a-zA-Z0-9_]*)\.([a-zA-Z_][a-zA-Z0-9_]*)',
        # spark.read.table / spark.table
        r'\.table\s*\(\s*["\']([a-zA-Z_][a-zA-Z0-9_]*)\.([a-zA-Z_][a-zA-Z0-9_]*)\.([a-zA-Z_][a-zA-Z0-9_]*)["\']',
        # USE CATALOG
        r'USE\s+CATALOG\s+([a-zA-Z_][a-zA-Z0-9_]*)',
        # spark.sql with catalog reference
        r'spark\.sql\s*\(["\'][^"]*\b([a-zA-Z_][a-zA-Z0-9_]*)\.([a-zA-Z_][a-zA-Z0-9_]*)\.([a-zA-Z_][a-zA-Z0-9_]*)\b',
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            if isinstance(match, tuple):
                catalog_name = match[0]
            else:
                catalog_name = match
            
            # Skip common false positives
            if catalog_name.lower() not in ['self', 'os', 'sys', 'json', 'pd', 'np', 'spark', 'df', 'data']:
                context = f"{filename}: {catalog_name}.{'.'.join(match[1:]) if isinstance(match, tuple) and len(match) > 1 else ''}"
                catalogs_found.append({
                    "catalog": catalog_name,
                    "file": filename,
                    "full_reference": '.'.join(match) if isinstance(match, tuple) else match
                })
    
    return catalogs_found

print("Helper functions ready")

## Export each app as a bundle

In [ ]:
exports = []
errors = []
all_catalog_refs = []

print("Exporting apps as bundles...")
print("=" * 60)

for app_name in app_names:
    print(f"\nExporting: {app_name}")
    app_catalog_refs = []
    
    try:
        app = get_app_via_rest(app_name)
        source_path = app.get('active_deployment', {}).get('source_code_path', '')
        description = app.get('description', '')
        
        bundle_path = f"{volume_path}/bundles/{app_name}"
        resources_path = f"{bundle_path}/resources"
        
        files_exported = 0
        
        # Export source files and scan for catalogs
        if source_path:
            objects = list_workspace_dir(source_path)
            for obj in objects:
                if obj.get('object_type') == 'FILE':
                    file_content = export_workspace_file(obj['path'])
                    if file_content:
                        file_name = obj['path'].split('/')[-1]
                        w.files.upload(f"{bundle_path}/{file_name}", BytesIO(file_content), overwrite=True)
                        files_exported += 1
                        
                        # Scan for catalog references in code files
                        if file_name.endswith(('.py', '.sql', '.yaml', '.yml', '.json')):
                            refs = scan_for_catalogs(file_content, file_name)
                            for ref in refs:
                                ref['app_name'] = app_name
                                app_catalog_refs.append(ref)
            
            print(f"  Exported {files_exported} source files")
            if app_catalog_refs:
                unique_catalogs = set(r['catalog'] for r in app_catalog_refs)
                print(f"  Found catalog references: {unique_catalogs}")
        
        # Generate databricks.yml with app_name as variable (defaults to original)
        databricks_yml = f'''bundle:
  name: {app_name}

variables:
  app_name:
    description: "App name (change for same-workspace testing)"
    default: "{app_name}"
  app_description:
    description: "App description"
    default: "{description}"

targets:
  source:
    workspace:
      host: {source_host}
    default: true

  target:
    workspace:
      host: {target_host}

include:
  - resources/*.yml
'''
        w.files.upload(f"{bundle_path}/databricks.yml", BytesIO(databricks_yml.encode('utf-8')), overwrite=True)
        
        # Generate resources/app.yml
        app_yml = f'''resources:
  apps:
    {safe_name(app_name)}:
      name: ${{var.app_name}}
      description: ${{var.app_description}}
      source_code_path: ../
'''
        w.files.upload(f"{resources_path}/app.yml", BytesIO(app_yml.encode('utf-8')), overwrite=True)
        
        # Write app-specific catalog_mapping.csv if catalogs found
        if app_catalog_refs:
            unique_catalogs = list(set(r['catalog'] for r in app_catalog_refs))
            mapping_content = "source_catalog,target_catalog,notes\n"
            for cat in sorted(unique_catalogs):
                mapping_content += f"{cat},,# Fill in target catalog\n"
            w.files.upload(f"{bundle_path}/catalog_mapping.csv", BytesIO(mapping_content.encode('utf-8')), overwrite=True)
            all_catalog_refs.extend(app_catalog_refs)
        
        exports.append({
            "app_name": app_name,
            "bundle_path": bundle_path,
            "files_exported": files_exported,
            "catalogs_found": len(set(r['catalog'] for r in app_catalog_refs)),
            "status": "SUCCESS",
        })
        print(f"  Bundle created at: {bundle_path}")
        
    except Exception as e:
        exports.append({
            "app_name": app_name,
            "bundle_path": "",
            "files_exported": 0,
            "catalogs_found": 0,
            "status": "FAILED",
        })
        errors.append(f"{app_name}: {e}")
        print(f"  FAILED: {e}")

print("\n" + "=" * 60)

## Write global catalog mapping

In [ ]:
# Write a global catalog_mapping.csv with all unique catalogs across all apps
if all_catalog_refs:
    unique_catalogs = sorted(set(r['catalog'] for r in all_catalog_refs))
    
    global_mapping_path = f"{volume_path}/catalog_mapping.csv"
    mapping_content = "source_catalog,target_catalog,notes\n"
    for cat in unique_catalogs:
        apps_using = sorted(set(r['app_name'] for r in all_catalog_refs if r['catalog'] == cat))
        mapping_content += f"{cat},,# Used by: {', '.join(apps_using)}\n"
    
    w.files.upload(global_mapping_path, BytesIO(mapping_content.encode('utf-8')), overwrite=True)
    print(f"Global catalog mapping: {global_mapping_path}")
    print(f"Catalogs found: {unique_catalogs}")
else:
    print("No catalog references detected in app code.")

## Write export manifest

In [ ]:
manifest_path = f"{volume_path}/export_manifest.csv"

output = StringIO()
writer = csv.DictWriter(output, fieldnames=["app_name", "bundle_path", "files_exported", "catalogs_found", "status"])
writer.writeheader()
writer.writerows(exports)

w.files.upload(manifest_path, BytesIO(output.getvalue().encode('utf-8')), overwrite=True)
print(f"Export manifest: {manifest_path}")

## Summary and Next Steps

In [ ]:
success_count = len([e for e in exports if e['status'] == 'SUCCESS'])
fail_count = len(errors)
catalogs_count = len(set(r['catalog'] for r in all_catalog_refs)) if all_catalog_refs else 0

print("=" * 60)
print("EXPORT SUMMARY")
print("=" * 60)
print(f"Total apps: {len(app_names)}")
print(f"Successful: {success_count}")
print(f"Failed: {fail_count}")
print(f"Unique catalogs found: {catalogs_count}")
print(f"\nBundles location: {volume_path}/bundles/")
print("")
print("NEXT STEPS:")
print("1. Download bundles: databricks fs cp -r dbfs:{volume_path}/bundles/<app> ./")
print("2. Fill in catalog_mapping.csv with target catalog names")
print("3. Run transform script: python transform_catalogs.py")
print("4. Deploy to target: databricks bundle deploy -t target")
print("")
print("For same-workspace testing (different app name):")
print("  databricks bundle deploy -t target --var app_name=<new-name>")
print("")
print("For cross-workspace migration (same app name):")
print("  databricks bundle deploy -t target")
print("=" * 60)

if errors:
    print("\nErrors:")
    for err in errors:
        print(f"  - {err}")

display(pd.DataFrame(exports))